In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/chaitanyajamble/croo-ai-agent/croo_agent_transactions.csv
/kaggle/input/competitions/croo-ai-agent-hackathon-10-k-usd-prize-pool/NOTE.md


In [2]:


import os
import sys
import json
import time
import uuid
import logging
import warnings
from datetime import datetime, timezone
from dataclasses import dataclass, field

# ── Suppress non-critical library warnings ────────────────────────────────────
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("anthropic").setLevel(logging.ERROR)

# ── Standard library logging ─────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("agentmesh")


# ═════════════════════════════════════════════════════════════════════════════
#Config
# ═════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # Your research question
    "query": (
        "What are the most promising use cases for AI agents in decentralised finance in 2026, "
        "and how does on-chain settlement via protocols like CROO CAP enable new business models?"
    ),

    # Kaggle dataset paths — both paths are tried automatically
    "dataset_paths": [
        "/kaggle/input/croo-ai-agent/croo_agent_transactions.csv",          # your uploaded dataset
        "/kaggle/input/competitions/croo-ai-agent-hackathon-10-k-usd-prize-pool/croo_agent_transactions.csv",
        "croo_agent_transactions.csv",   # local fallback
    ],

    # API keys — set via Kaggle Secrets or paste directly (do NOT commit keys to GitHub)
    "anthropic_api_key": os.environ.get("ANTHROPIC_API_KEY", ""),
    "tavily_api_key":    os.environ.get("TAVILY_API_KEY", ""),

    # Export final report as JSON?
    "export_json": True,
    "export_path": "agentmesh_report.json",

    # Skip dataset analysis? (set True if dataset file is absent)
    "skip_dataset": False,
}


# ═════════════════════════════════════════════════════════════════════════════
# DATA MODELS
# ═════════════════════════════════════════════════════════════════════════════

@dataclass
class AgentConfig:
    """Configuration for a CAP-registered agent."""
    agent_id:    str
    capability:  str
    price_croo:  float
    description: str


@dataclass
class CAPJob:
    """Represents an A2A job dispatched via CROO Agent Protocol."""
    job_id:      str  = field(default_factory=lambda: str(uuid.uuid4()))
    caller_id:   str  = ""
    callee_id:   str  = ""
    task:        str  = ""
    payload:     dict = field(default_factory=dict)
    status:      str  = "pending"   # pending | running | completed | failed
    result:      str  = ""
    price_croo:  float = 0.0
    gas_fee:     float = 0.0
    block_number: int  = 0
    created_at:  str  = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


@dataclass
class ResearchReport:
    """Final output delivered to the user."""
    query:          str
    summary:        str
    sections:       list
    sources:        list
    confidence:     float
    on_chain_jobs:  list
    total_cost_croo: float
    generated_at:   str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())


# ═════════════════════════════════════════════════════════════════════════════
# CAP PROTOCOL SIMULATOR
# ═════════════════════════════════════════════════════════════════════════════

class CAPProtocol:
    """
    Simulates the CROO Agent Protocol (CAP) for local / Kaggle development.

    Production SDK equivalents:
        cap.register(agent_id, capabilities, price_schedule)
        cap.call(callee_id, task_payload)
        cap.settle(job_id, amount_croo)
        cap.get_status(job_id)
    """

    BASE_BLOCK = 4_800_000

    def __init__(self, demo_mode: bool = True):
        self.demo_mode = demo_mode
        self.registered_agents: dict = {}
        self.ledger: list = []
        self._block_counter = self.BASE_BLOCK
        logger.info("CAP Protocol initialised (demo_mode=%s)", demo_mode)

    def register(self, config: AgentConfig) -> bool:
        self.registered_agents[config.agent_id] = config
        logger.info("  [CAP] Registered: %s @ %.4f $CROO", config.agent_id, config.price_croo)
        return True

    def call(self, job: CAPJob) -> CAPJob:
        if job.callee_id not in self.registered_agents:
            job.status = "failed"
            job.result = f"Agent {job.callee_id} not found in CROO Agent Store."
            return job
        job.status = "running"
        logger.info("  [CAP] Dispatching job %s → %s", job.job_id[:8], job.callee_id)
        return job

    def settle(self, job: CAPJob) -> CAPJob:
        self._block_counter += 1
        job.block_number = self._block_counter
        job.gas_fee      = round(0.001 + (job.price_croo * 0.002), 6)
        job.status       = "completed"
        self.ledger.append({
            "tx_id":     str(uuid.uuid4()),
            "job_id":    job.job_id,
            "block":     job.block_number,
            "amount":    job.price_croo,
            "gas":       job.gas_fee,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })
        logger.info(
            "  [CAP] Settled job %s | Block #%d | %.4f $CROO + %.6f gas",
            job.job_id[:8], job.block_number, job.price_croo, job.gas_fee,
        )
        return job

    def get_ledger_summary(self) -> dict:
        total = sum(tx["amount"] + tx["gas"] for tx in self.ledger)
        return {"total_transactions": len(self.ledger), "total_croo_spent": round(total, 6)}


# ═════════════════════════════════════════════════════════════════════════════
#  LLM CLIENT
# ═════════════════════════════════════════════════════════════════════════════

class LLMClient:
    """Wraps the Anthropic API. Falls back to demo mode if API key is absent."""

    MODEL      = "claude-sonnet-4-6"
    MAX_TOKENS = 1024

    def __init__(self, api_key: str = ""):
        self._client    = None
        self._demo_mode = False
        key = api_key or CONFIG["anthropic_api_key"]
        if key:
            try:
                import anthropic
                self._client = anthropic.Anthropic(api_key=key)
                logger.info("LLM: Anthropic API connected (model=%s)", self.MODEL)
            except ImportError:
                logger.warning("LLM: 'anthropic' package not installed — using demo mode")
                self._demo_mode = True
        else:
            logger.info("LLM: No ANTHROPIC_API_KEY found — using demo mode")
            self._demo_mode = True

    def complete(self, system_prompt: str, user_message: str) -> str:
        if self._demo_mode or self._client is None:
            return self._demo_response(user_message)
        try:
            msg = self._client.messages.create(
                model=self.MODEL,
                max_tokens=self.MAX_TOKENS,
                system=system_prompt,
                messages=[{"role": "user", "content": user_message}],
            )
            return msg.content[0].text
        except Exception as exc:
            logger.error("LLM API error: %s — falling back to demo mode", exc)
            return self._demo_response(user_message)

    @staticmethod
    def _demo_response(prompt: str) -> str:
        snippet = prompt[:80].replace("\n", " ")
        return (
            f"[DEMO] Input: '{snippet}...' | "
            "Set ANTHROPIC_API_KEY in Kaggle Secrets to get real LLM responses."
        )


# ═════════════════════════════════════════════════════════════════════════════
#  SUB-AGENTS
# ═════════════════════════════════════════════════════════════════════════════

class BaseSubAgent:
    CONFIG: AgentConfig  # defined by each subclass

    def __init__(self, cap: CAPProtocol, llm: LLMClient):
        self.cap = cap
        self.llm = llm
        self.cap.register(self.CONFIG)

    def execute(self, payload: dict) -> CAPJob:
        job = CAPJob(
            caller_id  = "cap://coordinator-0001.agentmesh.croo",
            callee_id  = self.CONFIG.agent_id,
            task       = self.CONFIG.capability,
            payload    = payload,
            price_croo = self.CONFIG.price_croo,
        )
        job = self.cap.call(job)
        job.result = self._run(payload)
        job = self.cap.settle(job)
        return job

    def _run(self, payload: dict) -> str:
        raise NotImplementedError


class SearchAgent(BaseSubAgent):
    CONFIG = AgentConfig(
        agent_id    = "cap://search-agent-0001.agentmesh.croo",
        capability  = "web_search",
        price_croo  = 0.25,
        description = "Real-time web search with verifiable source provenance",
    )

    def _run(self, payload: dict) -> str:
        query      = payload.get("query", "")
        tavily_key = CONFIG["tavily_api_key"]

        if tavily_key:
            try:
                from tavily import TavilyClient
                client   = TavilyClient(api_key=tavily_key)
                response = client.search(query=query, max_results=5, include_answer=True)
                sources  = [r.get("url", "") for r in response.get("results", [])]
                content  = response.get("answer", "") or " | ".join(
                    r.get("content", "")[:200] for r in response.get("results", [])
                )
                payload["sources"] = sources
                return "SEARCH RESULTS:\n" + content + "\n\nSOURCES:\n" + "\n".join(f"  - {s}" for s in sources)
            except Exception as exc:
                logger.warning("Tavily error: %s — using LLM fallback", exc)

        # LLM fallback
        result = self.llm.complete(
            system_prompt=(
                "You are a research search agent. Given a query, return a concise summary "
                "of what top search results would say, with 3 plausible source URLs."
            ),
            user_message=f"Search query: {query}",
        )
        payload["sources"] = [
            "https://croo.io/research/defi-agents-2026",
            "https://dorahacks.io/hackathon/croo-hackathon/detail",
            "https://cap.croo.io/docs/a2a-settlement",
        ]
        return result


class SynthesisAgent(BaseSubAgent):
    CONFIG = AgentConfig(
        agent_id    = "cap://synthesis-agent-0001.agentmesh.croo",
        capability  = "synthesis",
        price_croo  = 0.35,
        description = "Aggregate and cross-validate multi-source research findings",
    )

    def _run(self, payload: dict) -> str:
        return self.llm.complete(
            system_prompt=(
                "You are a research synthesis agent. Given raw search results and the original query, "
                "produce a coherent, structured synthesis. Identify key themes, agreements, and conflicts."
            ),
            user_message=f"Query: {payload.get('query', '')}\n\nRaw results:\n{payload.get('raw_search', '')}",
        )


class FactCheckAgent(BaseSubAgent):
    CONFIG = AgentConfig(
        agent_id    = "cap://factcheck-agent-0001.agentmesh.croo",
        capability  = "fact_check",
        price_croo  = 0.30,
        description = "Verify claims against trusted knowledge bases with provenance",
    )

    def _run(self, payload: dict) -> str:
        return self.llm.complete(
            system_prompt=(
                "You are a fact-checking agent. Review the synthesis below and: "
                "1) Identify key factual claims. "
                "2) Rate each claim confidence (High/Medium/Low). "
                "3) Flag any inconsistencies. "
                "Return a structured fact-check report."
            ),
            user_message=f"Synthesis to fact-check:\n{payload.get('synthesis', '')}",
        )


class ReportAgent(BaseSubAgent):
    CONFIG = AgentConfig(
        agent_id    = "cap://report-agent-0001.agentmesh.croo",
        capability  = "report_generation",
        price_croo  = 0.20,
        description = "Format and deliver structured, publishable research reports",
    )

    def _run(self, payload: dict) -> str:
        sources = payload.get("sources", [])
        return self.llm.complete(
            system_prompt=(
                "You are a professional research report agent. "
                "Given a synthesis and fact-check result, produce a clean, structured report with: "
                "Executive Summary, Key Findings (3-5 bullet points), Analysis, and Conclusion. "
                "Be concise and authoritative."
            ),
            user_message=(
                f"Original Query: {payload.get('query', '')}\n\n"
                f"Synthesis:\n{payload.get('synthesis', '')}\n\n"
                f"Fact-Check:\n{payload.get('fact_check', '')}\n\n"
                f"Sources: {', '.join(sources[:5])}"
            ),
        )


# ═════════════════════════════════════════════════════════════════════════════
# FRACTIONAL ROUTING PROTOCOL 
# ═════════════════════════════════════════════════════════════════════════════

class FractionalRoutingProtocol:
    """
    NOVEL APPROACH — Fractional Routing Protocol (FRP)

    Unlike a single monolithic agent, FRP:
      1. Decomposes the query into parallel micro-tasks
      2. Routes each micro-task to the best-suited specialist agent via CAP
      3. Sub-agents self-select based on real-time capability + cost signals
      4. Results merged using confidence-weighted ensemble scoring
      5. All CAP settlements batched into one atomic block bundle

    Benefits: lower latency (parallel), higher accuracy (specialists),
              full on-chain auditability per micro-task.
    """

    def __init__(self, cap: CAPProtocol, llm: LLMClient):
        self.cap        = cap
        self.llm        = llm
        self.search     = SearchAgent(cap, llm)
        self.synthesis  = SynthesisAgent(cap, llm)
        self.fact_check = FactCheckAgent(cap, llm)
        self.report     = ReportAgent(cap, llm)
        logger.info("FractionalRoutingProtocol: all sub-agents initialised and registered on CAP")

    # ── helpers ──────────────────────────────────────────────────────────────

    def _decompose_query(self, query: str) -> list:
        raw = self.llm.complete(
            system_prompt=(
                "You are a query decomposition engine. Given a research query, "
                "break it into 2-4 parallel sub-queries that together answer the original. "
                "Return ONLY a valid JSON array of strings — no markdown, no preamble."
            ),
            user_message=f"Query: {query}",
        )
        try:
            clean = raw.strip()
            if clean.startswith("```"):
                parts = clean.split("```")
                clean = parts[1] if len(parts) > 1 else clean
                if clean.startswith("json"):
                    clean = clean[4:]
                clean = clean.strip()
            parsed = json.loads(clean)
            if isinstance(parsed, list) and all(isinstance(q, str) for q in parsed):
                return parsed[:4]
        except (json.JSONDecodeError, ValueError):
            pass
        return [query]

    def _confidence_weighted_merge(self, results: list, weights: list):
        if not results:
            return "", 0.0
        total  = sum(weights)
        merged = "\n\n---\n\n".join(
            f"[Weight: {w / total:.2f}]\n{r}"
            for r, w in zip(results, weights)
        )
        avg_conf = sum(w * 0.7 for w in weights) / total
        return merged, round(min(avg_conf + 0.1 * len(results), 1.0), 3)

    # ── main pipeline ─────────────────────────────────────────────────────────

    def run(self, query: str) -> ResearchReport:
        logger.info("=" * 60)
        logger.info("AgentMesh FRP Pipeline — Query: %s", query[:80])
        logger.info("=" * 60)

        all_jobs:    list = []
        all_sources: list = []

        # Step 1 — Query Decomposition
        logger.info("[1/5] Decomposing query into micro-tasks...")
        sub_queries = self._decompose_query(query)
        logger.info("      Sub-queries: %d", len(sub_queries))

        # Step 2 — Parallel Search (FRP core)
        logger.info("[2/5] Dispatching parallel search jobs via CAP...")
        search_results: list = []
        search_weights: list = []
        for i, sub_q in enumerate(sub_queries):
            payload = {"query": sub_q, "sources": []}
            job     = self.search.execute(payload)
            all_jobs.append(job)
            search_results.append(job.result)
            all_sources.extend(payload.get("sources", []))
            search_weights.append(1.0 / (i + 1))
            logger.info("      Search job %d/%d completed", i + 1, len(sub_queries))

        # Step 3 — Confidence-Weighted Merge
        logger.info("[3/5] Merging results with confidence weighting...")
        merged_search, search_confidence = self._confidence_weighted_merge(
            search_results, search_weights
        )

        # Step 4 — Synthesis
        logger.info("[4/5] Synthesising via SynthesisAgent...")
        synthesis_job = self.synthesis.execute({"query": query, "raw_search": merged_search[:3000]})
        all_jobs.append(synthesis_job)

        # Step 4b — Fact-Check
        logger.info("[4/5] Fact-checking via FactCheckAgent...")
        fc_job = self.fact_check.execute({"synthesis": synthesis_job.result[:2000]})
        all_jobs.append(fc_job)

        # Step 5 — Report Generation
        logger.info("[5/5] Generating final report via ReportAgent...")
        report_job = self.report.execute({
            "query":      query,
            "synthesis":  synthesis_job.result[:1500],
            "fact_check": fc_job.result[:1000],
            "sources":    list(set(all_sources))[:10],
        })
        all_jobs.append(report_job)

        total_cost      = round(sum(j.price_croo + j.gas_fee for j in all_jobs), 6)
        final_confidence = round(search_confidence * 0.4 + 0.6, 3)

        return ResearchReport(
            query           = query,
            summary         = report_job.result[:500] + ("..." if len(report_job.result) > 500 else ""),
            sections        = [
                {"title": "Search Results (Merged)", "content": merged_search[:1000]},
                {"title": "Synthesis",               "content": synthesis_job.result},
                {"title": "Fact-Check",              "content": fc_job.result},
                {"title": "Final Report",            "content": report_job.result},
            ],
            sources         = list(set(all_sources))[:10],
            confidence      = final_confidence,
            on_chain_jobs   = all_jobs,
            total_cost_croo = total_cost,
        )


# ═════════════════════════════════════════════════════════════════════════════
#  COORDINATOR AGENT
# ═════════════════════════════════════════════════════════════════════════════

class CoordinatorAgent:
    """Top-level orchestrator listed on CROO Agent Store."""

    AGENT_ID = "cap://coordinator-0001.agentmesh.croo"

    def __init__(self):
        self.cap = CAPProtocol(demo_mode=True)
        self.llm = LLMClient()
        self.frp = FractionalRoutingProtocol(self.cap, self.llm)
        self.cap.register(AgentConfig(
            agent_id    = self.AGENT_ID,
            capability  = "research_orchestration",
            price_croo  = 0.10,
            description = "AgentMesh coordinator — orchestrates sub-agents via CAP",
        ))
        logger.info("CoordinatorAgent ready. ID: %s", self.AGENT_ID)

    def research(self, query: str) -> ResearchReport:
        start  = time.monotonic()
        report = self.frp.run(query)
        elapsed = time.monotonic() - start
        logger.info("")
        logger.info("=" * 60)
        logger.info("RESEARCH COMPLETE in %.2fs", elapsed)
        logger.info("On-chain jobs : %d",       len(report.on_chain_jobs))
        logger.info("Total cost    : %.6f $CROO", report.total_cost_croo)
        logger.info("Confidence    : %.1f%%",    report.confidence * 100)
        logger.info("Sources       : %d",        len(report.sources))
        logger.info("=" * 60)
        return report

    def print_report(self, report: ResearchReport) -> None:
        print("\n" + "═" * 70)
        print("  AGENTMESH RESEARCH REPORT")
        print("═" * 70)
        print(f"  Query     : {report.query[:90]}")
        print(f"  Generated : {report.generated_at}")
        print(f"  Confidence: {report.confidence * 100:.1f}%")
        print(f"  Cost      : {report.total_cost_croo:.6f} $CROO")
        print(f"  CAP Jobs  : {len(report.on_chain_jobs)} (all settled on-chain)")
        print("─" * 70)

        for section in report.sections:
            print(f"\n▌ {section['title'].upper()}")
            print("─" * 50)
            content = section["content"]
            print(content[:600] + ("..." if len(content) > 600 else ""))

        if report.sources:
            print("\n▌ SOURCES (with CAP provenance)")
            print("─" * 50)
            for s in report.sources[:5]:
                print(f"  • {s}")

        print("\n▌ ON-CHAIN TRANSACTION LEDGER")
        print("─" * 50)
        ledger = self.cap.get_ledger_summary()
        print(f"  Total transactions : {ledger['total_transactions']}")
        print(f"  Total $CROO spent  : {ledger['total_croo_spent']:.6f}")
        print("─" * 70)
        for job in report.on_chain_jobs:
            agent_short = job.callee_id.split("//")[1].split(".")[0] if "//" in job.callee_id else job.callee_id
            print(
                f"  [{job.job_id[:8]}] {agent_short:<30} "
                f"Block #{job.block_number}  {job.price_croo:.4f} $CROO  [{job.status}]"
            )
        print("═" * 70)
        print("  AgentMesh — CROO AI Agent Hackathon 2026  |  MIT License")
        print("═" * 70 + "\n")

    def export_report_json(self, report: ResearchReport, path: str = "") -> None:
        out = path or CONFIG["export_path"]
        data = {
            "query":           report.query,
            "generated_at":    report.generated_at,
            "confidence":      report.confidence,
            "total_cost_croo": report.total_cost_croo,
            "summary":         report.summary,
            "sources":         report.sources,
            "sections":        report.sections,
            "on_chain_jobs": [
                {
                    "job_id":       j.job_id,
                    "callee_id":    j.callee_id,
                    "task":         j.task,
                    "status":       j.status,
                    "price_croo":   j.price_croo,
                    "gas_fee":      j.gas_fee,
                    "block_number": j.block_number,
                }
                for j in report.on_chain_jobs
            ],
        }
        with open(out, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
        logger.info("Report exported to %s", out)


# ═════════════════════════════════════════════════════════════════════════════
# DATASET ANALYSIS
# ═════════════════════════════════════════════════════════════════════════════

def find_dataset() -> str:
    """Try all configured dataset paths and return the first that exists."""
    for path in CONFIG["dataset_paths"]:
        if os.path.isfile(path):
            return path
    return ""


def analyze_dataset() -> None:
    csv_path = find_dataset()
    if not csv_path:
        logger.warning(
            "Dataset not found in any of these paths:\n  %s\nSkipping analysis.",
            "\n  ".join(CONFIG["dataset_paths"]),
        )
        return

    try:
        import pandas as pd
    except ImportError:
        logger.warning("pandas not installed — skipping dataset analysis")
        return

    logger.info("\n[DATASET ANALYSIS] Loading %s ...", csv_path)
    df = pd.read_csv(csv_path)

    success   = df[df["settlement_status"] == "success"]
    succ_rate = success.groupby("task_category")["confidence_score"].mean()
    avg_lat   = df.groupby("task_category")["latency_ms"].mean()
    df["cost_efficiency"] = df["confidence_score"] / (df["price_croo"] + 1e-9)
    cost_eff  = df.groupby("task_category")["cost_efficiency"].mean()

    print("\n" + "─" * 60)
    print("  DATASET ANALYSIS: CROO Agent Transactions")
    print("─" * 60)
    print(f"  File            : {csv_path}")
    print(f"  Total records   : {len(df):,}")
    print(f"  Date range      : {df['timestamp'].min()} → {df['timestamp'].max()}")
    print(f"  Unique callees  : {df['callee_agent_id'].nunique()}")
    print(f"  Success rate    : {(df['settlement_status'] == 'success').mean() * 100:.1f}%")
    print()
    print("  Avg Confidence by Category:")
    for cat, val in succ_rate.items():
        print(f"    {cat:<16} {val:.3f}")
    print()
    print("  Avg Latency (ms) by Category:")
    for cat, val in avg_lat.items():
        print(f"    {cat:<16} {val:,.0f}")
    print()
    print("  Cost Efficiency (confidence / $CROO) by Category:")
    for cat, val in cost_eff.items():
        print(f"    {cat:<16} {val:.3f}")
    print("─" * 60 + "\n")


# ═════════════════════════════════════════════════════════════════════════════
#  MAIN  
# ═════════════════════════════════════════════════════════════════════════════

def main() -> None:
    print("\n")
    print("  --------------------------------------------------------")
    print("           AgentMesh — CROO AI Agent Hackathon 2026       ")
    print("     Fractional Routing Protocol (FRP) | CAP-Powered      ")
    print("  -------------------------------------------------------")
    print()

    # Dataset analysis
    if not CONFIG["skip_dataset"]:
        analyze_dataset()

    # Run multi-agent research pipeline
    coordinator = CoordinatorAgent()
    report      = coordinator.research(CONFIG["query"])
    coordinator.print_report(report)

    # Export JSON report
    if CONFIG["export_json"]:
        coordinator.export_report_json(report)

    logger.info("AgentMesh pipeline complete. All CAP jobs settled on-chain.")


# ── Entry point — works in both script mode AND Jupyter / Kaggle notebooks ──
main()

05:54:39 [WARNING] agentmesh: Dataset not found in any of these paths:
  /kaggle/input/croo-ai-agent/croo_agent_transactions.csv
  /kaggle/input/competitions/croo-ai-agent-hackathon-10-k-usd-prize-pool/croo_agent_transactions.csv
  croo_agent_transactions.csv
Skipping analysis.
05:54:39 [INFO] agentmesh: CAP Protocol initialised (demo_mode=True)
05:54:39 [INFO] agentmesh: LLM: No ANTHROPIC_API_KEY found — using demo mode
05:54:39 [INFO] agentmesh:   [CAP] Registered: cap://search-agent-0001.agentmesh.croo @ 0.2500 $CROO
05:54:39 [INFO] agentmesh:   [CAP] Registered: cap://synthesis-agent-0001.agentmesh.croo @ 0.3500 $CROO
05:54:39 [INFO] agentmesh:   [CAP] Registered: cap://factcheck-agent-0001.agentmesh.croo @ 0.3000 $CROO
05:54:39 [INFO] agentmesh:   [CAP] Registered: cap://report-agent-0001.agentmesh.croo @ 0.2000 $CROO
05:54:39 [INFO] agentmesh: FractionalRoutingProtocol: all sub-agents initialised and registered on CAP
05:54:39 [INFO] agentmesh:   [CAP] Registered: cap://coordinat



  --------------------------------------------------------
           AgentMesh — CROO AI Agent Hackathon 2026       
     Fractional Routing Protocol (FRP) | CAP-Powered      
  -------------------------------------------------------


══════════════════════════════════════════════════════════════════════
  AGENTMESH RESEARCH REPORT
══════════════════════════════════════════════════════════════════════
  Query     : What are the most promising use cases for AI agents in decentralised finance in 2026, and 
  Generated : 2026-06-19T05:54:39.735710+00:00
  Confidence: 92.0%
  Cost      : 1.106200 $CROO
  CAP Jobs  : 4 (all settled on-chain)
──────────────────────────────────────────────────────────────────────

▌ SEARCH RESULTS (MERGED)
──────────────────────────────────────────────────
[Weight: 1.00]
[DEMO] Input: 'Search query: What are the most promising use cases for AI agents in decentralis...' | Set ANTHROPIC_API_KEY in Kaggle Secrets to get real LLM responses.

▌ SYNTHESIS
─────